In [8]:
import sys
from pyprojroot import here
sys.path.insert(0, str(here()))

In [9]:
import copy
import optuna
import numpy as np
from transformers import AutoTokenizer
import time

from src.train import train_single_fold, tokenize_df
from src.config import load_config
from src.data_holdout import get_pooled_df, get_fold_from_disk

import yaml
import torch
from datasets import Dataset
from pyprojroot import here
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

from sklearn.metrics import f1_score
from peft import LoraConfig, get_peft_model, TaskType



In [12]:
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"

### Check names of linear layers

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

linear_names = []
for name, module in model.named_modules():
    if module.__class__.__name__ == "Linear":
        linear_names.append(name)

for n in linear_names:
    print(n)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1730.97it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                     

bert.encoder.layer.0.attention.self.query
bert.encoder.layer.0.attention.self.key
bert.encoder.layer.0.attention.self.value
bert.encoder.layer.0.attention.output.dense
bert.encoder.layer.0.intermediate.dense
bert.encoder.layer.0.output.dense
bert.encoder.layer.1.attention.self.query
bert.encoder.layer.1.attention.self.key
bert.encoder.layer.1.attention.self.value
bert.encoder.layer.1.attention.output.dense
bert.encoder.layer.1.intermediate.dense
bert.encoder.layer.1.output.dense
bert.encoder.layer.2.attention.self.query
bert.encoder.layer.2.attention.self.key
bert.encoder.layer.2.attention.self.value
bert.encoder.layer.2.attention.output.dense
bert.encoder.layer.2.intermediate.dense
bert.encoder.layer.2.output.dense
bert.encoder.layer.3.attention.self.query
bert.encoder.layer.3.attention.self.key
bert.encoder.layer.3.attention.self.value
bert.encoder.layer.3.attention.output.dense
bert.encoder.layer.3.intermediate.dense
bert.encoder.layer.3.output.dense
bert.encoder.layer.4.attention.s

### Test query, value vector only vs all linear layers

In [11]:
def run_quick_lora_test(target_modules, fold=0):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3
    )

    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        target_modules=target_modules,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
    )

    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir=str(here("results/tmp_lora_test")),
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=2e-4,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        weight_decay=0.01,
        warmup_ratio=0.1,
        num_train_epochs=5,
        metric_for_best_model="macro_f1",
        fp16=True,
        seed=42,
        logging_steps=10,
        report_to="none",
        skip_memory_metrics=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds[fold],
        eval_dataset=val_ds[fold],
        compute_metrics=compute_metrics,
    )

    start = time.time()
    trainer.train()
    metrics = trainer.evaluate()
    elapsed = time.time() - start

    del trainer, model
    torch.cuda.empty_cache()

    return {
        "target_modules": target_modules,
        "runtime_min": elapsed / 60,
        "macro_f1": metrics["eval_macro_f1"],
    }

results_qv = run_quick_lora_test(["query","value"])
results_all = run_quick_lora_test("all-linear")

print(results_qv)
print(results_all)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3070.98it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                   

trainable params: 297,219 || all params: 109,781,766 || trainable%: 0.2707


Epoch,Training Loss,Validation Loss,Macro F1
1,0.961530,0.938684,0.253411
2,0.860614,0.945006,0.253411
3,0.894376,0.928840,0.253411
4,0.843265,0.929298,0.253411
5,0.991721,0.931361,0.253411


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3456.30it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                   

trainable params: 1,341,699 || all params: 110,826,246 || trainable%: 1.2106


Epoch,Training Loss,Validation Loss,Macro F1
1,0.947932,0.953899,0.253411
2,0.831557,0.983508,0.253411
3,0.817474,0.949330,0.281800
4,0.720484,0.956825,0.330089
5,0.817736,0.969245,0.310102


{'target_modules': ['query', 'value'], 'runtime_min': 0.38967015345891315, 'macro_f1': 0.253411306042885}
{'target_modules': 'all-linear', 'runtime_min': 1.075599523385366, 'macro_f1': 0.31010233770318696}


## Optuna Experiment

In [13]:

# open shared settings
with open(here("configs/base.yaml"), "r") as f:
    base_data = yaml.full_load(f)

# get df
full_df = get_pooled_df()

# same pattern as collaborator: all folds
train_folds, val_folds = get_fold_from_disk(full_df)

# load tokenizer and write tokenize function
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fold(df):
    ds = Dataset.from_pandas(df, preserve_index=False)

    def _tokenize(examples):
        return tokenizer(
            examples["abstract_conclusion"],
            examples["press_release_conclusion"],
            truncation=True,
            padding="max_length",
            max_length=base_data["max_length"],
        )

    cols_to_remove = [c for c in ds.column_names if c != "exaggeration_label"]
    ds = ds.map(_tokenize, batched=True, remove_columns=cols_to_remove)
    ds = ds.rename_column("exaggeration_label", "labels")
    ds.set_format("torch")
    return ds

# tokenize all folds
train_ds = [tokenize_fold(df) for df in train_folds]
val_ds = [tokenize_fold(df) for df in val_folds]

for i in range(len(train_ds)):
    print(f"Fold {i}: train={len(train_ds[i])} val={len(val_ds[i])}")

# define metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

# def build_lora_model():
#     model = AutoModelForSequenceClassification.from_pretrained(
#         MODEL_NAME,
#         num_labels=3
#     )

#     # LoRA applied broadly to linear layers
#     peft_config = LoraConfig(
#         task_type=TaskType.SEQ_CLS,
#         target_modules="all-linear",
#         r=16,                # placeholder, overwritten per trial below
#         lora_alpha=32,       # placeholder, overwritten per trial below
#         lora_dropout=0.05,   # placeholder, overwritten per trial below
#         bias="none",
#     )

#     model = get_peft_model(model, peft_config)
#     return model

# set up Optuna objective
def objective(trial):
    folds_to_run = [0, 1, 3] # folds chosen based on previous experiments

    # LoRA-specific search space
    lr = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    batch_size = trial.suggest_categorical("per_device_train_batch_size", [8])
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.05, 0.2)
    num_epochs = 8

    lora_r = trial.suggest_categorical("lora_r", [2, 4, 8, 16])
    lora_alpha = trial.suggest_categorical("lora_alpha", [4, 8, 16, 32])
    lora_dropout = trial.suggest_categorical("lora_dropout", [0.0, 0.05, 0.1])

    f1_scores = []
    fold_times_min = []

    for fold in folds_to_run:
        fold_start = time.time()
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=3
        )

        peft_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            target_modules=["query", "key", "value", "dense"], #approximation of all linear, get same results  in testing
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            bias="none",
        )

        model = get_peft_model(model, peft_config)

        # confirm correct number of trainable parameters
        if fold == 0:
            model.print_trainable_parameters()

        training_args = TrainingArguments(
            output_dir=str(here(f"results/optuna/pubmedbert_lora2/trial-{trial.number}/fold-{fold}")),
            eval_strategy=base_data["eval_strategy"],
            save_strategy="no",              # no checkpoints this time
            learning_rate=lr,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            weight_decay=weight_decay,
            warmup_ratio=warmup_ratio,
            num_train_epochs=num_epochs,
            metric_for_best_model="macro_f1",
            fp16=True,
            seed=base_data["seed"],
            logging_steps=10,
            report_to=base_data["report_to"],
            skip_memory_metrics=True,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds[fold],
            eval_dataset=val_ds[fold],
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=10)],
        )

        trainer.train()
        eval_metrics = trainer.evaluate()
        f1_scores.append(eval_metrics["eval_macro_f1"])


        fold_time_min = (time.time() - fold_start) / 60
        fold_times_min.append(fold_time_min)
        print(
            f"Trial {trial.number} | Fold {fold} | "
            f"F1={eval_metrics['eval_macro_f1']:.4f} | "
            f"Time={fold_time_min:.2f} min"
        )

        del trainer
        del model
        torch.cuda.empty_cache()

    return float(np.mean(f1_scores))

# study definition
db_path = here("results/optuna/optuna_study_pubmedbert_lora.db")
storage_url = f"sqlite:///{db_path}"
n_startup_trials = 5

print(storage_url)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        n_startup_trials=n_startup_trials,
        seed=42,
    ),
    study_name="pubmedbert-lora-hyperparameter-search2",
    storage=storage_url,
    load_if_exists=True
)

study.optimize(objective, n_trials=50)

# best trials
print(f"\n{'='*60}")
print(f"  Best macro_f1: {study.best_trial.value:.4f}")
print(f"  Best params:   {study.best_trial.params}")
print(f"{'='*60}")

/workspace/exaggeration-detection/src/data_holdout.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(ENCODED_LABELS)
Map: 100%|██████████| 106/106 [00:00<00:00, 12470.79 examples/s]


Fold 0: train=424 val=106
Fold 1: train=424 val=106
Fold 2: train=424 val=106
Fold 3: train=424 val=106
Fold 4: train=424 val=106
sqlite:////workspace/exaggeration-detection/results/optuna/optuna_study_pubmedbert_lora.db


[I 2026-03-23 20:55:40,002] A new study created in RDB with name: pubmedbert-lora-hyperparameter-search2
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1740.97it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEX

trainable params: 337,155 || all params: 109,821,702 || trainable%: 0.3070


Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


...

Trial 49 | Fold 1 | F1=0.4117 | Time=0.66 min


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3147.51it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                     

Epoch,Training Loss,Validation Loss,Macro F1
1,0.884933,0.941155,0.253411
2,1.028252,0.933749,0.253411
3,0.898268,0.953598,0.253411
4,0.509923,1.079842,0.433152
5,0.546617,1.309387,0.388710
6,0.531541,1.647160,0.430152
7,0.182593,1.863787,0.356902
8,0.172947,2.047732,0.406412


[I 2026-03-23 22:29:06,499] Trial 49 finished with value: 0.39869825128107483 and parameters: {'learning_rate': 0.00022997365248092854, 'per_device_train_batch_size': 8, 'weight_decay': 0.05855646746732814, 'warmup_ratio': 0.17128672200758693, 'lora_r': 2, 'lora_alpha': 32, 'lora_dropout': 0.1}. Best is trial 36 with value: 0.4541717885905509.


Trial 49 | Fold 3 | F1=0.4064 | Time=0.66 min

  Best macro_f1: 0.4542
  Best params:   {'learning_rate': 0.0003755537454177177, 'per_device_train_batch_size': 8, 'weight_decay': 0.03500575278073154, 'warmup_ratio': 0.18809558678514623, 'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.1}


In [14]:

print(study.best_trial)
print(study.best_value)
print(study.best_params)

FrozenTrial(number=36, state=<TrialState.COMPLETE: 1>, values=[0.4541717885905509], datetime_start=datetime.datetime(2026, 3, 23, 22, 1, 30, 892187), datetime_complete=datetime.datetime(2026, 3, 23, 22, 3, 28, 931786), params={'learning_rate': 0.0003755537454177177, 'per_device_train_batch_size': 8, 'weight_decay': 0.03500575278073154, 'warmup_ratio': 0.18809558678514623, 'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'learning_rate': FloatDistribution(high=0.0005, log=True, low=1e-05, step=None), 'per_device_train_batch_size': CategoricalDistribution(choices=(8,)), 'weight_decay': FloatDistribution(high=0.1, log=False, low=0.0, step=None), 'warmup_ratio': FloatDistribution(high=0.2, log=False, low=0.05, step=None), 'lora_r': CategoricalDistribution(choices=(2, 4, 8, 16)), 'lora_alpha': CategoricalDistribution(choices=(4, 8, 16, 32)), 'lora_dropout': CategoricalDistribution(choices=(0.0, 0.05, 0.1))}, trial_i